In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [6]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [7]:
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 10)

In [8]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape)
print(test.shape)

(15000, 24)
(10000, 23)


In [9]:
train.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,smoking
0,0,40.0,170.0,75.0,93.0,1.2,1.2,1.0,1.0,114.0,...,43.0,95.0,16.3,1.0,0.9,23.0,19.0,83.0,0.0,1.0
1,1,45.0,155.0,55.0,83.0,1.0,1.0,1.0,1.0,116.0,...,67.0,86.0,13.5,1.0,0.7,17.0,16.0,11.0,0.0,0.0
2,2,40.0,165.0,70.0,83.0,1.2,1.2,1.0,1.0,122.0,...,47.0,92.0,15.2,1.0,0.9,19.0,29.0,17.0,1.0,1.0
3,3,40.0,175.0,80.0,89.5,1.5,1.2,1.0,1.0,110.0,...,47.0,103.0,15.7,1.0,0.9,21.0,20.0,23.0,0.0,1.0
4,4,35.0,175.0,95.0,98.0,0.8,0.8,1.0,1.0,147.0,...,46.0,96.0,17.0,1.0,1.2,33.0,56.0,149.0,0.0,1.0


In [10]:
train.isnull().sum()

,0
id,0
age,0
height(cm),0
weight(kg),0
waist(cm),0
eyesight(left),0
eyesight(right),0
hearing(left),0
hearing(right),0
systolic,0


In [11]:
test.isnull().sum()

,0
id,0
age,0
height(cm),0
weight(kg),0
waist(cm),0
eyesight(left),0
eyesight(right),0
hearing(left),0
hearing(right),0
systolic,0


In [12]:
def feature_engineering(df):
    df['BMI'] = df['weight(kg)'] / ((df['height(cm)']/100) ** 2)
    df['pulse_pressure'] = df['systolic'] - df['relaxation']
    df['chol_ratio'] = df['Cholesterol'] / df['HDL']
    df['ldl_hdl_ratio'] = df['LDL'] / df['HDL']
    df['ast_alt_ratio'] = df['AST'] / (df['ALT'] + 1)
    df['waist_height_ratio'] = df['waist(cm)'] / df['height(cm)']
    return df

train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

In [13]:
train_fe.head()

,id,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,ALT,Gtp,dental caries,smoking,BMI,pulse_pressure,chol_ratio,ldl_hdl_ratio,ast_alt_ratio,waist_height_ratio
0,0,40.0,170.0,75.0,93.0,1.2,1.2,1.0,1.0,114.0,...,19.0,83.0,0.0,1.0,25.951557,40.0,4.046512,2.209302,1.150000,0.547059
1,1,45.0,155.0,55.0,83.0,1.0,1.0,1.0,1.0,116.0,...,16.0,11.0,0.0,0.0,22.892820,44.0,2.537313,1.283582,1.000000,0.535484
2,2,40.0,165.0,70.0,83.0,1.2,1.2,1.0,1.0,122.0,...,29.0,17.0,1.0,1.0,25.711662,44.0,3.340426,1.957447,0.633333,0.503030
3,3,40.0,175.0,80.0,89.5,1.5,1.2,1.0,1.0,110.0,...,20.0,23.0,0.0,1.0,26.122449,40.0,3.723404,2.191489,1.000000,0.511429
4,4,35.0,175.0,95.0,98.0,0.8,0.8,1.0,1.0,147.0,...,56.0,149.0,0.0,1.0,31.020408,54.0,3.782609,2.086957,0.578947,0.560000


In [14]:
train_fe = train_fe.drop(['id'], axis = 1, errors='ignore')
test_fe = test_fe.drop(['id'],  axis = 1, errors='ignore')

In [15]:
x_train = train_fe.drop(['smoking'], axis=1)
y_train = train_fe['smoking']
x_test = test_fe

In [16]:
# model = XGBClassifier(n_estimators=500, learning_rate=0.5)
model = XGBClassifier(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)
model_cv_score = cross_val_score(model, x_train, y_train, cv = 5, scoring = 'accuracy')
print(model_cv_score)
print(model_cv_score.mean())
model.fit(x_train, y_train)

[0.814      0.80366667 0.80166667 0.81066667 0.809     ]
0.8078


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.02, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1200,
              n_jobs=None, num_parallel_tree=None, ...)

In [17]:
prediction = model.predict(x_test)
print(f"{prediction.mean():.2%}")

39.28%


In [18]:
print(len(prediction))

10000


In [19]:
importance = pd.DataFrame({
    'feature': x_train.columns,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(importance.head(15))

               feature  importance
1           height(cm)    0.272615
15          hemoglobin    0.129768
20                 Gtp    0.048738
12        triglyceride    0.043652
21       dental caries    0.038234
17    serum creatinine    0.032278
0                  age    0.030516
22                 BMI    0.023186
19                 ALT    0.022946
14                 LDL    0.022545
2           weight(kg)    0.022181
18                 AST    0.021850
11         Cholesterol    0.021626
27  waist_height_ratio    0.021525
4       eyesight(left)    0.020183


In [20]:
probs = model.predict_proba(x_test)[:, 1]

In [21]:
submission = pd.DataFrame(
    {
        'id' : test['id'],
        'smoking' : probs
    }
)
submission.to_csv('submission.csv', index = False)

In [24]:
from lightgbm import LGBMClassifier
lgb_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=50,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8
)
lgb_model_cv_score = cross_val_score(lgb_model, x_train, y_train, cv = 5, scoring = 'accuracy')
print(lgb_model_cv_score)
print(lgb_model_cv_score.mean())
lgb_model.fit(x_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 4380, number of negative: 7620
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004418 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2592
[LightGBM] [Info] Number of data points in the train set: 12000, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.365000 -> initscore=-0.553728
[LightGBM] [Info] Start training from score -0.553728
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 4380, number of negative: 7620
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006195 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2598
[LightGBM] [Info] N

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.03, n_estimators=800,
               num_leaves=50, subsample=0.8)

In [25]:
lgb_prediction = lgb_model.predict(x_test)
print(f"{lgb_prediction.mean():.2%}")

39.37%


In [26]:
importance = pd.DataFrame({
    'feature': x_train.columns,
    'importance': lgb_model.feature_importances_
}).sort_values(by='importance', ascending=False)

print(importance.head(15))

                feature  importance
12         triglyceride        2892
15           hemoglobin        2294
20                  Gtp        2267
26        ast_alt_ratio        2223
10  fasting blood sugar        2122
11          Cholesterol        2012
27   waist_height_ratio        2004
14                  LDL        1865
25        ldl_hdl_ratio        1856
24           chol_ratio        1692
3             waist(cm)        1659
8              systolic        1569
19                  ALT        1562
9            relaxation        1509
18                  AST        1501


In [27]:
lgb_probs = lgb_model.predict_proba(x_test)[:, 1]

In [28]:
submission = pd.DataFrame(
    {
        'id' : test['id'],
        'smoking' : lgb_probs
    }
)
submission.to_csv('lgb_submission.csv', index = False)